# Quando uma descoberta científica é justificável?

Experimento reproduzível de identificação ativa de mecanismos, com ruído, intervenções restritas, custos e regras de declaração. **Versão de pesquisa 1.0 — 01/10/2026.**

O notebook é autossuficiente: ele cria o código, gera os dados, compara oito métodos, salva checkpoints e exporta as tabelas. Não há API paga nem avaliação humana. A100 é um alvo de execução, mas o Colab gratuito não garante esse hardware.

**Escopo:** identificação em um catálogo finito, não descoberta livre de toda equação possível. A garantia vale para o catálogo e ruído especificados; os testes de estresse são reportados separadamente. O objetivo do piloto é descobrir se a contribuição candidata merece ser ampliada. Não existe promessa de resultado ou aceitação top tier.

Use **Ambiente de execução → Alterar tipo de ambiente → GPU**, e execute as células em ordem. O padrão `pilot` é a primeira execução científica exploratória. O Drive é opcional, mas necessário para recuperação após perda do runtime. O painel LLM só começa depois que o núcleo termina.


In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ.setdefault('MPLCONFIGDIR','/tmp/discovery_matplotlib')
# Keep Colab's working PyTorch/CUDA installation. Install only missing core packages.
packages={'numpy':'numpy>=1.26', 'scipy':'scipy>=1.12', 'pandas':'pandas>=2.2',
          'matplotlib':'matplotlib>=3.8', 'torch':'torch>=2.4'}
missing=[requirement for module,requirement in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install',*missing],check=True)
import torch
print('Python:',sys.version.split()[0], '| PyTorch:',torch.__version__)
if torch.cuda.is_available():
    props=torch.cuda.get_device_properties(0)
    print('GPU:',props.name,'| memória:',round(props.total_memory/2**30,1),'GiB')
    torch.backends.cuda.matmul.allow_tf32=False
else:
    print('Sem CUDA. Use PRESET="smoke" para a verificação em CPU.')


In [ ]:
# Embedded, inspectable source.
EMBEDDED_FILES = {'discovery.py': '"""Controlled active mechanism identification. No LLM judge or hidden-law policy input.\n\nStatistical claim applies ONLY to the declared finite catalog and specified Gaussian\nobservation model. This is a research pilot, not unrestricted equation discovery.\n"""\nfrom __future__ import annotations\nimport argparse, dataclasses, hashlib, itertools, json, math, os, platform, time\nfrom dataclasses import dataclass\nfrom fractions import Fraction as F\nfrom pathlib import Path\nimport numpy as np\nimport torch\n\nVERSION = \'1.0.0\'\nMETHODS = (\'random_safe\', \'variance_safe\', \'eig_safe\', \'chernoff_safe\',\n           \'deficit_safe\', \'eig_posterior\', \'deficit_posterior\', \'eig_fixed\')\n\ndef digest(obj):\n    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()\n\ndef seed_for(*parts):\n    return int(digest(parts)[:15], 16) % (2**32-1)\n\ndef atomic_json(path, obj):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix+\'.tmp\')\n    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False))\n    os.replace(tmp, path)\n\n@dataclass(frozen=True)\nclass Config:\n    preset: str = \'pilot\'\n    seed: int = 20261001\n    budget: int = 60\n    alpha: float = .05\n    batch: int = 32\n    eig_samples: int = 12\n    game_iterations: int = 32\n    device: str = \'auto\'\n    trace_per_group: int = 2\n    max_minutes: float = 90\n    methods: tuple = METHODS\n\ndef settings(preset):\n    return {\n        \'smoke\': dict(repeats=1, per_family=1, noises=(.15,), stress=False),\n        \'pilot\': dict(repeats=8, per_family=2, noises=(.08,.20), stress=True),\n        \'confirmatory\': dict(repeats=100, per_family=2, noises=(.08,.20), stress=True),\n    }[preset]\n\ndef value(domain, family, a, b, x, z):\n    """Works with exact rational numbers as well as float inputs."""\n    if domain == \'mechanics\':\n        return {\n            \'linear\': lambda: a*x,\n            \'additive\': lambda: a*x+b*z,\n            \'cubic\': lambda: a*x+b*x**3,\n            \'interaction\': lambda: a*x+b*x*z,\n            \'squared\': lambda: a*x*x+b*z*z,\n            \'mixed\': lambda: a*x*x+b*z,\n        }[family]()\n    return {\n        \'michaelis_menten\': lambda: a*x/(b+x),\n        \'competitive\': lambda: a*x/(b*(1+z)+x),\n        \'uncompetitive\': lambda: a*x/(b+x*(1+z)),\n        \'noncompetitive\': lambda: a*x/((b+x)*(1+z)),\n        \'substrate_inhibition\': lambda: a*x/(b+x+x*x),\n        \'hill2\': lambda: a*x*x/(b*b+x*x),\n    }[family]()\n\ndef expression(domain, family, a, b):\n    forms = {\n        \'linear\': \'{a}*x\', \'additive\': \'{a}*x+{b}*z\',\n        \'cubic\': \'{a}*x+{b}*x**3\', \'interaction\': \'{a}*x+{b}*x*z\',\n        \'squared\': \'{a}*x**2+{b}*z**2\', \'mixed\': \'{a}*x**2+{b}*z\',\n        \'michaelis_menten\': \'{a}*x/({b}+x)\',\n        \'competitive\': \'{a}*x/({b}*(1+z)+x)\',\n        \'uncompetitive\': \'{a}*x/({b}+x*(1+z))\',\n        \'noncompetitive\': \'{a}*x/(({b}+x)*(1+z))\',\n        \'substrate_inhibition\': \'{a}*x/({b}+x+x**2)\',\n        \'hill2\': \'{a}*x**2/({b}**2+x**2)\',\n    }\n    return forms[family].format(a=f\'({a})\',b=f\'({b})\')\n\ndef make_catalog(domain, regime=\'full\', remove=None):\n    families = ((\'linear\',\'additive\',\'cubic\',\'interaction\',\'squared\',\'mixed\')\n                if domain == \'mechanics\' else (\'michaelis_menten\',\'competitive\',\n                \'uncompetitive\',\'noncompetitive\',\'substrate_inhibition\',\'hill2\'))\n    axis = [F(-2),F(-1),F(-1,2),F(1,2),F(1),F(2)] if domain==\'mechanics\' else [F(1,4),F(1,2),F(1),F(2),F(4)]\n    zs = axis if domain==\'mechanics\' else [F(0),F(1,4),F(1,2),F(1),F(2)]\n    full_actions = list(itertools.product(axis,zs))\n    actions = [p for p in full_actions if regime==\'full\' or\n               (p[1]==p[0] if domain==\'mechanics\' else p[1]==0)]\n    models=[]; exact=[]\n    for family in families:\n        if family == remove: continue\n        for a,b in itertools.product((F(1,2),F(1),F(2)), repeat=2):\n            if family==\'linear\' and b!=1: continue  # remove unused-parameter duplicates\n            models.append(dict(family=family,a=str(a),b=str(b),formula=expression(domain,family,a,b)))\n            exact.append(tuple(value(domain,family,a,b,x,z) for x,z in actions))\n    # Exact equivalence on the COMPLETE, publicly specified finite intervention menu.\n    # No tolerance-based numerical clustering, and no assertion beyond this menu.\n    signatures={}; groups=[]\n    for sig in exact:\n        if sig not in signatures: signatures[sig]=len(signatures)\n        groups.append(signatures[sig])\n    fnames=sorted({m[\'family\'] for m in models})\n    family=np.array([fnames.index(m[\'family\']) for m in models])\n    prior=np.array([1/(len(fnames)*sum(family==f)) for f in family])\n    costs=np.array([1 if (z==x if domain==\'mechanics\' else z==0) else 3 for x,z in actions])\n    # Public, fixed units; never normalize by hidden ground-truth coefficients.\n    scale=4.0 if domain==\'mechanics\' else 1.0\n    mu=np.array([[float(v)/scale for v in row] for row in exact])\n    return dict(domain=domain,regime=regime,models=models,actions=actions,\n                exact=exact,mu=mu,family=family,fnames=fnames,groups=np.array(groups),\n                prior=prior,costs=costs,scale=scale)\n\ndef build_cases(cfg):\n    s=settings(cfg.preset); cases=[]\n    for domain in (\'mechanics\',\'kinetics\'):\n        full=make_catalog(domain)\n        selected=[]\n        for fam in full[\'fnames\']:\n            ids=[i for i,m in enumerate(full[\'models\']) if m[\'family\']==fam]\n            rng=np.random.default_rng(seed_for(cfg.seed,domain,fam,\'selection\'))\n            selected.extend(rng.choice(ids,size=min(s[\'per_family\'],len(ids)),replace=False).tolist())\n        contexts=[(r,\'closed\') for r in (\'full\',\'restricted\')]\n        if s[\'stress\']:\n            contexts += [(\'full\',\'off_grid\'),(\'full\',\'noise_mismatch\'),(\'full\',\'missing_family\')]\n        for regime,scenario in contexts:\n            for noise in s[\'noises\']:\n                for truth_index in selected:\n                    model=full[\'models\'][truth_index]\n                    for rep in range(s[\'repeats\']):\n                        base=dict(domain=domain,regime=regime,scenario=scenario,\n                                  noise=noise,truth_index=truth_index,rep=rep,seed=cfg.seed)\n                        base[\'id\']=digest(base)[:24]\n                        base[\'cluster\']=f\'{domain}:{truth_index}\'\n                        base[\'truth_family\']=model[\'family\']\n                        cases.append(base)\n    return cases\n\ndef case_group(case):\n    return (case[\'domain\'],case[\'regime\'],case[\'scenario\'],case[\'noise\'],\n            case[\'truth_family\'] if case[\'scenario\']==\'missing_family\' else \'\')\n\ndef prepare_group(cases):\n    c=cases[0]; remove=c[\'truth_family\'] if c[\'scenario\']==\'missing_family\' else None\n    bank=make_catalog(c[\'domain\'],c[\'regime\'],remove)\n    full=make_catalog(c[\'domain\'])\n    truth=[]; signatures=[]\n    for case in cases:\n        m=full[\'models\'][case[\'truth_index\']]; a,b=F(m[\'a\']),F(m[\'b\'])\n        if case[\'scenario\']==\'off_grid\': a*=F(113,100); b*=F(107,100)\n        sig=tuple(value(c[\'domain\'],m[\'family\'],a,b,x,z) for x,z in bank[\'actions\'])\n        signatures.append(sig); truth.append([float(v)/bank[\'scale\'] for v in sig])\n    bank[\'noise\']=float(c[\'noise\'])\n    return bank,np.array(truth),signatures\n\ndef get_device(name=\'auto\'):\n    return torch.device((\'cuda\' if torch.cuda.is_available() else \'cpu\') if name==\'auto\' else name)\n\ndef tensors(bank,device):\n    f=lambda x: torch.as_tensor(x,device=device,dtype=torch.float64)\n    return dict(mu=f(bank[\'mu\']),prior=f(bank[\'prior\']),cost=f(bank[\'costs\']),\n                family=torch.as_tensor(bank[\'family\'],device=device),\n                groups=torch.as_tensor(bank[\'groups\'],device=device))\n\ndef update_evidence(ll, max_e, increment, prior):\n    ll=ll+increment\n    log_q=torch.logsumexp(ll+prior.log(),dim=-1)\n    log_e=log_q[:,None]-ll\n    return ll,torch.maximum(max_e,log_e)\n\ndef partition_mass(p,labels):\n    out=torch.zeros((len(p),int(labels.max())+1),dtype=p.dtype,device=p.device)\n    out.scatter_add_(1,labels.expand(len(p),-1),p)\n    return out\n\ndef choose_action(policy,ll,max_e,t,noise,allowed,uniform,mc_u,mc_z,cfg):\n    """Only public candidates + past observations enter this function."""\n    mu,cost,prior=t[\'mu\'],t[\'cost\'],t[\'prior\']\n    p=torch.softmax(ll+prior.log(),dim=-1)\n    b,m=ll.shape; a=mu.shape[1]\n    if policy==\'random\':\n        score=torch.zeros_like(allowed,dtype=mu.dtype)\n    elif policy==\'variance\':\n        score=((p @ mu.square())-(p @ mu).square()).clamp_min(0)/cost\n    elif policy==\'eig\':\n        # Monte Carlo mutual information about catalog atom, not claimed to be\n        # a reproduction of an external paper\'s Bayesian design implementation.\n        idx=torch.searchsorted(p.cumsum(-1).contiguous(),mc_u.contiguous()).clamp_max(m-1)\n        ys=mu[idx]+noise*mc_z[:,:,None]\n        logp=-.5*((ys[:,:,:,None]-mu.T[None,None,:,:])/noise).square()\n        mix=torch.logsumexp(logp+p.clamp_min(1e-300).log()[:,None,None,:],dim=-1)\n        score=(-.5*mc_z[:,:,None].square()-mix).mean(1)/cost\n    elif policy in (\'chernoff\',\'deficit\'):\n        leader=(ll+prior.log()).argmax(-1)\n        d=.5*((mu[leader,None,:]-mu[None,:,:])/noise).square()\n        # Do not try to separate indistinguishable atoms on this intervention menu.\n        rivals=t[\'groups\'][None,:]!=t[\'groups\'][leader,None]\n        if policy==\'deficit\':\n            rivals &= max_e < math.log(1/cfg.alpha)\n            gap=(math.log(1/cfg.alpha)-(torch.logsumexp(ll+prior.log(),-1)[:,None]-ll)).clamp_min(.25)\n            d=d/gap[:,:,None]\n        # Multiplicative-weights zero-sum game: maximize worst-rival separation\n        # per unit experimental cost. Approximate mixed design, no optimality claim.\n        d=d/cost[None,None,:]\n        scale=d.amax((1,2),keepdim=True).clamp_min(1e-12)\n        game=d/scale\n        rival_weight=rivals.to(mu.dtype)\n        allocation=torch.zeros((b,a),dtype=mu.dtype,device=mu.device)\n        for _ in range(cfg.game_iterations):\n            q=rival_weight/rival_weight.sum(-1,keepdim=True).clamp_min(1e-30)\n            scores=torch.einsum(\'bm,bma->ba\',q,game).masked_fill(~allowed,-torch.inf)\n            best=scores.argmax(-1)\n            allocation.scatter_add_(1,best[:,None],torch.ones((b,1),dtype=mu.dtype,device=mu.device))\n            gain=game.gather(2,best[:,None,None].expand(-1,m,1)).squeeze(-1)\n            rival_weight *= torch.exp(-2*gain)\n        # allocation describes spending fractions; convert to action frequencies.\n        probs=(allocation/cost)*allowed\n        probs=probs/probs.sum(-1,keepdim=True).clamp_min(1e-30)\n        fallback=allowed.to(mu.dtype)/allowed.sum(-1,keepdim=True).clamp_min(1)\n        probs=torch.where(rivals.any(-1)[:,None],probs,fallback)\n        probs=.98*probs+.02*fallback  # declared exploration, shared for both policies\n        return torch.searchsorted(probs.cumsum(-1).contiguous(),uniform[:,None].contiguous()).squeeze(-1).clamp_max(a-1)\n    else: raise ValueError(policy)\n    score=score.masked_fill(~allowed,-torch.inf)\n    # Random tie-breaking; separate policy randomness from observation noise.\n    ties=score==score.max(-1,keepdim=True).values\n    probs=ties.to(mu.dtype)/ties.sum(-1,keepdim=True)\n    return torch.searchsorted(probs.cumsum(-1).contiguous(),uniform[:,None].contiguous()).squeeze(-1).clamp_max(a-1)\n\ndef declaration(ll,max_e,t,cfg,rule):\n    b=len(ll); device=ll.device\n    kind=torch.zeros(b,dtype=torch.long,device=device) # 0 unresolved,1 family,2 class,3 empty\n    answer=torch.full((b,),-1,dtype=torch.long,device=device)\n    if rule==\'safe\':\n        alive=max_e < math.log(1/cfg.alpha)\n        families=partition_mass(alive.to(ll.dtype),t[\'family\'])>0\n        groups=partition_mass(alive.to(ll.dtype),t[\'groups\'])>0\n        single_family=families.sum(-1)==1\n        single_group=(groups.sum(-1)==1)&~single_family\n        kind[single_family]=1; answer[single_family]=families.long().argmax(-1)[single_family]\n        kind[single_group]=2; answer[single_group]=groups.long().argmax(-1)[single_group]\n        kind[~alive.any(-1)]=3\n    else:\n        p=torch.softmax(ll+t[\'prior\'].log(),dim=-1)\n        fm=partition_mass(p,t[\'family\']); gm=partition_mass(p,t[\'groups\'])\n        sf=fm.max(-1).values >= 1-cfg.alpha\n        sg=(gm.max(-1).values >= 1-cfg.alpha)&~sf\n        kind[sf]=1; answer[sf]=fm.argmax(-1)[sf]\n        kind[sg]=2; answer[sg]=gm.argmax(-1)[sg]\n    return kind,answer\n\ndef run_batch(cases,method,cfg,advisor=None):\n    bank,truth,signatures=prepare_group(cases)\n    device=get_device(cfg.device); t=tensors(bank,device); b,m=len(cases),len(bank[\'models\'])\n    policy,rule=method.rsplit(\'_\',1)\n    ll=torch.zeros((b,m),device=device,dtype=torch.float64); max_e=torch.zeros_like(ll)\n    cost=torch.zeros(b,device=device,dtype=torch.float64); steps=torch.zeros(b,device=device,dtype=torch.long)\n    kind=torch.zeros_like(steps); answer=torch.full_like(steps,-1)\n    stopped=torch.zeros(b,device=device,dtype=torch.bool)\n    noise=bank[\'noise\']; truth_t=torch.tensor(truth,device=device,dtype=torch.float64)\n    rows_idx=torch.arange(b,device=device)\n    # Random tapes are keyed by episode, so chunk size / resume cannot change draws.\n    eps=np.stack([np.random.default_rng(seed_for(c[\'id\'],\'noise\')).normal(size=cfg.budget) for c in cases])\n    u=np.stack([np.random.default_rng(seed_for(c[\'id\'],policy,\'policy\')).random(size=cfg.budget) for c in cases])\n    mc_u=np.stack([np.random.default_rng(seed_for(c[\'id\'],\'eig_u\')).random((cfg.budget,cfg.eig_samples)) for c in cases])\n    mc_z=np.stack([np.random.default_rng(seed_for(c[\'id\'],\'eig_z\')).normal(size=(cfg.budget,cfg.eig_samples)) for c in cases])\n    eps,u,mc_u,mc_z=[torch.tensor(z,device=device,dtype=torch.float64) for z in (eps,u,mc_u,mc_z)]\n    elapsed0=time.perf_counter(); traces=[]; history=[]; invalid_llm=0\n    true_matches=torch.tensor([[sig==s for s in bank[\'exact\']] for sig in signatures],device=device)\n    lost_truth=torch.zeros(b,dtype=torch.bool,device=device)\n    noise_factor=2 if cases[0][\'scenario\']==\'noise_mismatch\' else 1\n    for step in range(cfg.budget):\n        allowed=t[\'cost\'][None,:] <= (cfg.budget-cost[:,None])\n        active=~stopped & allowed.any(-1)\n        if not active.any(): break\n        allowed[~active,0]=True\n        if policy==\'llm\':\n            if b!=1 or advisor is None: raise ValueError(\'LLM panel requires single episode and advisor\')\n            fallback=int(choose_action(\'eig\',ll,max_e,t,noise,allowed,u[:,step],mc_u[:,step],mc_z[:,step],cfg)[0])\n            act,valid=advisor.choose(bank,ll[0].detach().cpu().numpy(),history,\n                                      allowed[0].cpu().numpy(),fallback)\n            invalid_llm+=int(not valid); action=torch.tensor([act],device=device)\n        else:\n            action=choose_action(policy,ll,max_e,t,noise,allowed,u[:,step],mc_u[:,step],mc_z[:,step],cfg)\n        means=t[\'mu\'][:,action].T\n        y=truth_t[rows_idx,action]+noise*noise_factor*eps[:,step]\n        inc=-.5*((y[:,None]-means)/noise).square() # identical normalizing constants cancel\n        ll_new,emax_new=update_evidence(ll,max_e,inc,t[\'prior\'])\n        ll=torch.where(active[:,None],ll_new,ll); max_e=torch.where(active[:,None],emax_new,max_e)\n        cost+=active*t[\'cost\'][action]; steps+=active\n        # Used ONLY for evaluation, never supplied to choose_action / advisor.\n        lost_truth |= active & true_matches.any(-1) & ~((max_e < math.log(1/cfg.alpha)) & true_matches).any(-1)\n        if rule!=\'fixed\':\n            new_kind,new_answer=declaration(ll,max_e,t,cfg,rule)\n            done=active&(new_kind>0)\n            kind[done]=new_kind[done]; answer[done]=new_answer[done]; stopped|=done\n        if policy==\'llm\': history.append(dict(action=int(action[0]),y=float(y[0])))\n        for i in range(min(cfg.trace_per_group,b)):\n            if bool(active[i]): traces.append(dict(id=cases[i][\'id\'],method=method,step=step+1,\n                action=int(action[i]),y=float(y[i]),cost=float(cost[i]),\n                survivors=int((max_e[i]<math.log(1/cfg.alpha)).sum()),decision=int(kind[i])))\n    if rule==\'fixed\':\n        p=torch.softmax(ll+t[\'prior\'].log(),-1)\n        kind[:]=1; answer=partition_mass(p,t[\'family\']).argmax(-1)\n    if device.type==\'cuda\': torch.cuda.synchronize()\n    wall=time.perf_counter()-elapsed0\n    out=[]\n    for i,c in enumerate(cases):\n        k=int(kind[i]); ans=int(answer[i]); correct=False; description=None\n        if k==1:\n            description=bank[\'fnames\'][ans]; correct=description==c[\'truth_family\']\n        elif k==2:\n            ids=np.flatnonzero(bank[\'groups\']==ans)\n            description=sorted({bank[\'models\'][j][\'family\'] for j in ids})\n            correct=signatures[i]==bank[\'exact\'][ids[0]]\n        resolved=k in (1,2)\n        out.append(dict(**c,method=method,decision={0:\'unresolved\',1:\'family\',2:\'equivalence\',3:\'empty_catalog\'}[k],\n            answer=description,resolved=resolved,correct=bool(correct),false_declaration=bool(resolved and not correct),\n            cost=float(cost[i]),queries=int(steps[i]),guarantee_applicable=c[\'scenario\']==\'closed\',\n            true_mean_in_catalog=bool(true_matches[i].any()),true_mean_eliminated=bool(lost_truth[i]),\n            surviving_atoms=int((max_e[i]<math.log(1/cfg.alpha)).sum()),\n            batch_seconds=wall,batch_size=b,amortized_seconds=wall/b,\n            llm_invalid_responses=invalid_llm if policy==\'llm\' else 0))\n    return out,traces\n\ndef environment():\n    return dict(python=platform.python_version(),torch=torch.__version__,numpy=np.__version__,\n                cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\n                vram_gib=round(torch.cuda.get_device_properties(0).total_memory/2**30,2) if torch.cuda.is_available() else None)\n\ndef load_rows(outdir):\n    rows=[]\n    for p in sorted((Path(outdir)/\'episodes\').glob(\'*.json\')):\n        data=json.loads(p.read_text())\n        rows.extend(data if isinstance(data,list) else [data])\n    keys=[(r[\'id\'],r[\'method\']) for r in rows]\n    if len(keys)!=len(set(keys)): raise ValueError(\'Episódios duplicados nos checkpoints.\')\n    return rows\n\ndef run_suite(cfg,outdir,checkpoint=None):\n    if not 0<cfg.alpha<1 or cfg.budget<1 or cfg.batch<1:\n        raise ValueError(\'Alpha, orçamento ou batch inválido.\')\n    outdir=Path(outdir); outdir.mkdir(parents=True,exist_ok=True)\n    spec=dataclasses.asdict(cfg)\n    # Operational settings may change across resumptions without changing experiment.\n    for k in (\'batch\',\'device\',\'max_minutes\',\'trace_per_group\'): spec.pop(k)\n    source_sha=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n    run_hash=digest(dict(spec=spec,source=source_sha))\n    manifest_path=outdir/\'manifest.json\'\n    if manifest_path.exists():\n        old=json.loads(manifest_path.read_text())\n        if old[\'run_hash\']!=run_hash: raise ValueError(\'Configuração/código diferente. Escolha outra pasta de execução.\')\n    else:\n        atomic_json(manifest_path,dict(run_hash=run_hash,config=dataclasses.asdict(cfg),\n            source_sha256=source_sha,environment=environment(),version=VERSION,created_unix=time.time()))\n    groups={}\n    for c in build_cases(cfg): groups.setdefault(case_group(c),[]).append(c)\n    total=sum(map(len,groups.values()))*len(cfg.methods)\n    existing={(r[\'id\'],r[\'method\']) for r in load_rows(outdir)}\n    start=time.monotonic(); done=0; completed=True\n    for key,cases in groups.items():\n        for method in cfg.methods:\n            # Atomic shards avoid thousands of tiny writes to mounted storage.\n            pending=[c for c in cases if (c[\'id\'],method) not in existing]\n            done+=len(cases)-len(pending)\n            for pos in range(0,len(pending),cfg.batch):\n                if cfg.max_minutes and time.monotonic()-start>cfg.max_minutes*60:\n                    completed=False; break\n                chunk=pending[pos:pos+cfg.batch]\n                result,traces=run_batch(chunk,method,cfg)\n                shard=digest([method,[c[\'id\'] for c in chunk]])[:24]\n                atomic_json(outdir/\'episodes\'/f\'{shard}.json\',result)\n                existing.update((r[\'id\'],r[\'method\']) for r in result)\n                if traces:\n                    atomic_json(outdir/\'traces\'/f"{chunk[0][\'id\']}_{method}.json",traces)\n                done+=len(chunk)\n                print(f\'{done}/{total} | {key[0]} {key[1]} {key[2]} | {method}\',flush=True)\n                if checkpoint: checkpoint(outdir)\n            if not completed: break\n        if not completed: break\n    atomic_json(outdir/\'status.json\',dict(completed=completed,expected_episodes=total,\n        completed_episodes=len(existing),environment=environment()))\n    if checkpoint: checkpoint(outdir,force=True)\n    print(\'Concluído.\' if completed else \'Pausa com checkpoint. Execute esta célula novamente para continuar.\')\n    return outdir\n\ndef summarize(outdir):\n    import pandas as pd\n    from scipy.stats import beta\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    outdir=Path(outdir); rows=load_rows(outdir)\n    if not rows: raise ValueError(\'Nenhum episódio salvo.\')\n    df=pd.DataFrame(rows)\n    df.to_csv(outdir/\'episodes.csv\',index=False)\n    keys=[\'domain\',\'regime\',\'scenario\',\'noise\',\'method\']\n    summary=df.groupby(keys).agg(n=(\'id\',\'size\'),resolution=(\'resolved\',\'mean\'),\n        correct_resolution=(\'correct\',\'mean\'),false_declarations=(\'false_declaration\',\'mean\'),\n        mean_cost=(\'cost\',\'mean\'),mean_queries=(\'queries\',\'mean\'),\n        truth_elimination=(\'true_mean_eliminated\',\'mean\')).reset_index()\n    summary.to_csv(outdir/\'summary.csv\',index=False)\n    # Exact binomial intervals per fixed truth and context, independent noise seeds.\n    risk=[]\n    for key,g in df.groupby(keys+[\'truth_index\']):\n        n=len(g); k=int(g.false_declaration.sum())\n        risk.append(dict(zip(keys+[\'truth_index\'],key),n=n,errors=k,\n            rate=k/n,lower_95=0.0 if k==0 else float(beta.ppf(.025,k,n-k+1)),\n            upper_95=1.0 if k==n else float(beta.ppf(.975,k+1,n-k))))\n    pd.DataFrame(risk).to_csv(outdir/\'risk_by_instance.csv\',index=False)\n    # Paired, law-cluster bootstrap: pairing by episode id, clustering by law.\n    paired=[]; baseline=\'eig_safe\'\n    for scenario in sorted(df.scenario.unique()):\n        sub=df[df.scenario==scenario]\n        for metric in [\'false_declaration\',\'correct\',\'cost\']:\n            wide=sub.pivot(index=[\'id\',\'cluster\'],columns=\'method\',values=metric)\n            if baseline not in wide: continue\n            for method in wide.columns:\n                if method==baseline: continue\n                x=wide[[baseline,method]].dropna().astype(float)\n                if x.empty: continue\n                delta=(x[method]-x[baseline]).groupby(level=\'cluster\').mean().to_numpy()\n                rng=np.random.default_rng(seed_for(scenario,metric,method,\'bootstrap\'))\n                samples=rng.choice(delta,size=(2000,len(delta)),replace=True).mean(-1)\n                paired.append(dict(scenario=scenario,metric=metric,method=method,baseline=baseline,\n                    paired_episodes=len(x),law_clusters=len(delta),difference=float(delta.mean()),\n                    lower_95=float(np.quantile(samples,.025)),upper_95=float(np.quantile(samples,.975))))\n    pd.DataFrame(paired).to_csv(outdir/\'paired_comparisons.csv\',index=False)\n    closed=df[df.scenario==\'closed\']\n    fig,ax=plt.subplots(1,2,figsize=(12,4),layout=\'constrained\')\n    agg=closed.groupby(\'method\')[[\'false_declaration\',\'correct\',\'cost\']].mean().sort_index()\n    if len(agg):\n        ax[0].barh(agg.index,agg.false_declaration)\n        alpha=json.loads((outdir/\'manifest.json\').read_text())[\'config\'][\'alpha\']\n        ax[0].axvline(alpha,color=\'red\',ls=\'--\',label=f\'nominal alpha={alpha}\')\n        ax[0].set_xlabel(\'False declarations / all episodes\'); ax[0].legend()\n        for method,g in closed.groupby(\'method\'):\n            xs=np.arange(0,int(g.cost.max())+1)\n            ax[1].plot(xs,[((g.correct)&(g.cost<=x)).mean() for x in xs],label=method)\n        ax[1].set(xlabel=\'Experimental cost\',ylabel=\'Correct resolution / all episodes\',ylim=(0,1))\n        ax[1].legend(fontsize=7)\n    fig.suptitle(\'Finite catalog, specified noise — exploratory aggregates\')\n    fig.savefig(outdir/\'main_results.png\',dpi=180); fig.savefig(outdir/\'main_results.pdf\'); plt.close(fig)\n    status=json.loads((outdir/\'status.json\').read_text()) if (outdir/\'status.json\').exists() else {}\n    text=(f\'# Relatório da execução\\n\\nEpisódios disponíveis: {len(df)}. \'\n          f\'Execução completa: {status.get("completed",False)}.\\n\\n\'\n          \'Os resultados medem identificação em um catálogo finito. Não demonstram descoberta irrestrita de equações. \'\n          \'A garantia é aplicável somente ao cenário closed, com o ruído especificado. \'\n          \'Os cenários off_grid, missing_family e noise_mismatch são testes de estresse sem essa garantia.\\n\\n\'\n          \'summary.csv contém resultados por contexto. risk_by_instance.csv fornece intervalos binomiais por lei. \'\n          \'paired_comparisons.csv usa bootstrap pareado por lei; valores negativos de cost/error favorecem o método. \'\n          \'Custo menor deve ser interpretado junto com resolução correta, pois uma recusa precoce também custa pouco.\\n\\n\'\n          \'O gráfico é descritivo e não substitui os intervalos por instância. Uma amostra pequena sem erros \'\n          \'não confirma que o risco é inferior a 5%. Nenhuma vitória estatística é declarada automaticamente.\\n\')\n    (outdir/\'REPORT.md\').write_text(text)\n    return summary\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(); p.add_argument(\'--preset\',default=\'smoke\',choices=[\'smoke\',\'pilot\',\'confirmatory\'])\n    p.add_argument(\'--out\',default=\'results\'); p.add_argument(\'--device\',default=\'auto\')\n    p.add_argument(\'--budget\',type=int,default=60); p.add_argument(\'--minutes\',type=float,default=90)\n    args=p.parse_args(); cfg=Config(preset=args.preset,device=args.device,budget=args.budget,max_minutes=args.minutes)\n    run_suite(cfg,args.out); print(summarize(args.out).to_string(index=False))\n', 'llm_panel.py': '"""Optional open-weight LLM acquisition panel. The LLM never grades discoveries."""\nimport gc, hashlib, json, re, time\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom discovery import (Config, atomic_json, build_cases, case_group, digest,\n                       environment, run_batch, seed_for)\n\nclass Advisor:\n    def __init__(self,logdir,model_id=\'Qwen/Qwen2.5-Coder-7B-Instruct\',revision=None):\n        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n        from huggingface_hub import model_info\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'O painel LLM requer GPU. O experimento principal funciona em CPU.\')\n        self.model_id=model_id\n        self.revision=revision or model_info(model_id).sha\n        self.logdir=Path(logdir); self.logdir.mkdir(parents=True,exist_ok=True)\n        self.tokenizer=AutoTokenizer.from_pretrained(model_id,revision=self.revision)\n        kwargs=dict(revision=self.revision,device_map=\'auto\',trust_remote_code=False)\n        free,total=torch.cuda.mem_get_info()\n        if free/2**30 >= 23:\n            kwargs[\'torch_dtype\']=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n            self.precision=str(kwargs[\'torch_dtype\'])\n        else:\n            kwargs[\'quantization_config\']=BitsAndBytesConfig(load_in_4bit=True,\n                bnb_4bit_compute_dtype=torch.float16,bnb_4bit_quant_type=\'nf4\',\n                bnb_4bit_use_double_quant=True)\n            self.precision=\'4-bit NF4\'\n        self.model=AutoModelForCausalLM.from_pretrained(model_id,**kwargs).eval()\n        self.calls=0; self.tokens=0; self.seconds=0.0\n\n    def choose(self,bank,ll,history,allowed,fallback):\n        # Public information only. No truth index, simulator seed or evaluation score.\n        order=np.argsort(ll+np.log(bank[\'prior\']))[-8:][::-1]\n        hypotheses=[dict(expression=bank[\'models\'][int(i)][\'formula\'],\n                         relative_log_likelihood=round(float(ll[i]-max(ll)),3)) for i in order]\n        actions=[dict(id=i,x=float(x),z=float(z),cost=int(bank[\'costs\'][i]))\n                 for i,(x,z) in enumerate(bank[\'actions\']) if allowed[i]]\n        prompt=("Choose the next experiment to distinguish the candidate mechanisms efficiently. "\n                "Measurement noise is independent Gaussian with known standard deviation. "\n                "An observation equals the formula divided by the supplied scale plus noise. "\n                "The hypothesis list is a likelihood-ranked shortlist, not guaranteed complete. "\n                "Use previous observations and differences between hypotheses. Return ONLY "\n                "a JSON object with the integer action id: {\\"action\\": 0}.\\n"+\n                json.dumps(dict(hypotheses=hypotheses,actions=actions,\n                                observations=history,scale=bank[\'scale\'],sigma=bank[\'noise\'])))\n        key=digest(dict(prompt=prompt,revision=self.revision,precision=self.precision))\n        path=self.logdir/f\'{key}.json\'\n        if path.exists():\n            saved=json.loads(path.read_text()); return saved[\'action\'],saved[\'valid\']\n        messages=[dict(role=\'system\',content=\'You design informative scientific experiments.\'),\n                  dict(role=\'user\',content=prompt)]\n        text=self.tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)\n        inputs=self.tokenizer(text,return_tensors=\'pt\').to(self.model.device)\n        if inputs[\'input_ids\'].shape[1]>12000:\n            raise RuntimeError(\'Prompt excessivo; reduza o orçamento do painel.\')\n        started=time.perf_counter()\n        with torch.inference_mode():\n            output=self.model.generate(**inputs,max_new_tokens=96,do_sample=False,\n                                       pad_token_id=self.tokenizer.eos_token_id)\n        torch.cuda.synchronize()\n        seconds=time.perf_counter()-started\n        new=output[0,inputs[\'input_ids\'].shape[1]:]\n        response=self.tokenizer.decode(new,skip_special_tokens=True)\n        match=re.search(r\'"action"\\s*:\\s*(\\d+)\',response)\n        action=int(match.group(1)) if match else -1\n        valid=0<=action<len(allowed) and bool(allowed[action])\n        if not valid: action=fallback  # recorded explicitly, never silently counted as pure LLM\n        atomic_json(path,dict(action=action,valid=valid,response=response,prompt=prompt,\n            model=self.model_id,revision=self.revision,precision=self.precision,\n            input_tokens=int(inputs[\'input_ids\'].numel()),output_tokens=len(new),seconds=seconds))\n        self.calls+=1; self.tokens+=len(new); self.seconds+=seconds\n        return action,valid\n\n    def close(self):\n        del self.model; gc.collect(); torch.cuda.empty_cache()\n\ndef run_panel(outdir,max_cases=12,budget=24,seed=20261001,checkpoint=None):\n    """Balanced small panel; exploratory only. Includes paired non-LLM controls."""\n    outdir=Path(outdir); outdir.mkdir(parents=True,exist_ok=True)\n    cfg=Config(preset=\'smoke\',seed=seed,budget=budget,trace_per_group=1,max_minutes=0)\n    # Deterministic stratified interleave avoids selecting only the first domain.\n    all_cases=build_cases(cfg); strata={}\n    for c in all_cases: strata.setdefault((c[\'domain\'],c[\'regime\']),[]).append(c)\n    cases=[]\n    for i in range(max(map(len,strata.values()))):\n        for key in sorted(strata):\n            if i<len(strata[key]): cases.append(strata[key][i])\n    cases=cases[:max_cases]\n    manifest=outdir/\'manifest.json\'\n    previous=json.loads(manifest.read_text()) if manifest.exists() else None\n    revision=previous[\'specification\'][\'revision\'] if previous else None\n    advisor=Advisor(outdir/\'llm_calls\',revision=revision)\n    specification=dict(model=advisor.model_id,revision=advisor.revision,precision=advisor.precision,\n        seed=seed,budget=budget,max_cases=max_cases,ids=[c[\'id\'] for c in cases],\n        core_sha=hashlib.sha256(Path(__file__).with_name(\'discovery.py\').read_bytes()).hexdigest(),\n        panel_sha=hashlib.sha256(Path(__file__).read_bytes()).hexdigest())\n    fp=digest(specification)\n    if manifest.exists() and json.loads(manifest.read_text())[\'fingerprint\']!=fp:\n        advisor.close(); raise ValueError(\'Painel diferente: use outra pasta para preservar a reprodução.\')\n    atomic_json(manifest,dict(fingerprint=fp,specification=specification,environment=environment()))\n    try:\n        for i,c in enumerate(cases):\n            for method in (\'llm_safe\',\'eig_safe\',\'deficit_safe\'):\n                path=outdir/\'episodes\'/f"{c[\'id\']}_{method}.json"\n                if path.exists(): continue\n                rows,trace=run_batch([c],method,cfg,advisor)\n                atomic_json(path,rows[0]); atomic_json(outdir/\'traces\'/path.name,trace)\n                if checkpoint: checkpoint(outdir.parent)\n            print(f\'Painel LLM: {i+1}/{len(cases)}\',flush=True)\n    finally: advisor.close()\n    import pandas as pd\n    df=pd.DataFrame([json.loads(p.read_text()) for p in sorted((outdir/\'episodes\').glob(\'*.json\'))])\n    df.to_csv(outdir/\'llm_panel.csv\',index=False)\n    atomic_json(outdir/\'status.json\',dict(completed=True,cases=len(cases),\n        note=\'Painel exploratório pequeno; não juntar à inferência confirmatória. Fallbacks estão registrados.\'))\n    return df.groupby(\'method\').agg(n=(\'id\',\'size\'),resolved=(\'resolved\',\'mean\'),\n        correct=(\'correct\',\'mean\'),false_declarations=(\'false_declaration\',\'mean\'),cost=(\'cost\',\'mean\'),\n        invalid_llm=(\'llm_invalid_responses\',\'sum\'))\n', 'test_discovery.py': '"""Scientific invariants: exact equivalence, sequential evidence, isolation, resume."""\nimport dataclasses, json, math, tempfile, unittest\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport discovery as d\n\ntorch.set_num_threads(1)\n\nclass ScientificTests(unittest.TestCase):\n    def test_exact_equivalence_is_restriction_dependent(self):\n        restricted=d.make_catalog(\'kinetics\',\'restricted\'); full=d.make_catalog(\'kinetics\',\'full\')\n        def idx(b,f): return next(i for i,m in enumerate(b[\'models\']) if m[\'family\']==f and m[\'a\']==\'1\' and m[\'b\']==\'1\')\n        i,j=idx(restricted,\'michaelis_menten\'),idx(restricted,\'competitive\')\n        self.assertEqual(restricted[\'exact\'][i],restricted[\'exact\'][j])\n        self.assertEqual(restricted[\'groups\'][i],restricted[\'groups\'][j])\n        self.assertNotEqual(full[\'exact\'][idx(full,\'michaelis_menten\')],full[\'exact\'][idx(full,\'competitive\')])\n\n    def test_no_duplicate_global_atoms_and_prior_normalized(self):\n        for dom in (\'mechanics\',\'kinetics\'):\n            b=d.make_catalog(dom)\n            self.assertEqual(len(b[\'exact\']),len(set(b[\'exact\'])))\n            self.assertAlmostEqual(sum(b[\'prior\']),1.)\n            self.assertTrue(np.isfinite(b[\'mu\']).all())\n\n    def test_log_evidence_matches_independent_likelihood_product(self):\n        prior=torch.tensor([.2,.3,.5],dtype=torch.float64)\n        inc=torch.tensor([[[-.4,-1.2,-.9]],[[-1.2,-.1,-.8]],[[-.7,-.8,-.1]]],dtype=torch.float64)\n        ll=torch.zeros((1,3),dtype=torch.float64); mx=ll.clone(); likelihood=np.ones(3); refmax=np.ones(3)\n        for step in inc:\n            ll,mx=d.update_evidence(ll,mx,step,prior)\n            likelihood*=np.exp(step.numpy()[0]); q=(prior.numpy()*likelihood).sum()\n            refmax=np.maximum(refmax,q/likelihood)\n            np.testing.assert_allclose(mx.numpy()[0],np.log(refmax),atol=1e-13)\n\n    def test_identical_distributions_cannot_be_separated_by_safe_rule(self):\n        b=d.make_catalog(\'kinetics\',\'restricted\'); t=d.tensors(b,\'cpu\')\n        cfg=d.Config(alpha=.05)\n        ll=torch.zeros((1,len(b[\'models\'])),dtype=torch.float64); mx=ll.clone()\n        true=next(i for i,m in enumerate(b[\'models\']) if m[\'family\']==\'competitive\' and m[\'a\']==\'1\' and m[\'b\']==\'1\')\n        for _ in range(40):\n            for a in range(len(b[\'actions\'])):\n                y=t[\'mu\'][true,a]\n                ll,mx=d.update_evidence(ll,mx,-.5*((y-t[\'mu\'][:,a])/.01).square()[None,:],t[\'prior\'])\n        kind,ans=d.declaration(ll,mx,t,cfg,\'safe\')\n        self.assertEqual(int(kind[0]),2)\n        self.assertEqual(int(ans[0]),int(b[\'groups\'][true]))\n\n    def test_policies_respect_cost_mask(self):\n        b=d.make_catalog(\'mechanics\'); t=d.tensors(b,\'cpu\'); m=len(b[\'models\'])\n        ll=torch.zeros((3,m),dtype=torch.float64); mx=ll.clone()\n        allowed=(t[\'cost\']<=1).expand(3,-1)\n        for policy in (\'random\',\'variance\',\'eig\',\'chernoff\',\'deficit\'):\n            a=d.choose_action(policy,ll,mx,t,.15,allowed,torch.tensor([.1,.5,.9],dtype=torch.float64),\n                torch.full((3,4),.3,dtype=torch.float64),torch.zeros((3,4),dtype=torch.float64),d.Config(game_iterations=8))\n            self.assertTrue(bool(allowed[torch.arange(3),a].all()))\n\n    def test_chunk_size_does_not_change_experiment(self):\n        cfg=d.Config(preset=\'smoke\',budget=12,eig_samples=4,game_iterations=8,device=\'cpu\')\n        cases=[c for c in d.build_cases(cfg) if c[\'domain\']==\'kinetics\' and c[\'regime\']==\'full\'][:2]\n        for method in (\'eig_safe\',\'deficit_safe\'):\n            together,_=d.run_batch(cases,method,cfg)\n            apart=[d.run_batch([c],method,cfg)[0][0] for c in cases]\n            for x,y in zip(together,apart):\n                for k in [\'decision\',\'answer\',\'cost\',\'queries\',\'correct\',\'surviving_atoms\']:\n                    self.assertEqual(x[k],y[k])\n\n    def test_resume_and_configuration_guard(self):\n        cfg=d.Config(preset=\'smoke\',budget=2,eig_samples=2,game_iterations=2,\n                     methods=(\'random_safe\',),device=\'cpu\',max_minutes=0)\n        with tempfile.TemporaryDirectory() as td:\n            d.run_suite(cfg,td); first=d.load_rows(td)\n            d.run_suite(dataclasses.replace(cfg,batch=1),td)\n            self.assertEqual(first,d.load_rows(td))\n            with self.assertRaises(ValueError): d.run_suite(dataclasses.replace(cfg,alpha=.1),td)\n\n    def test_guarantee_flag_does_not_extend_to_stress(self):\n        cfg=d.Config(preset=\'pilot\',budget=2,eig_samples=2,game_iterations=2,device=\'cpu\')\n        for scenario in (\'closed\',\'off_grid\',\'missing_family\',\'noise_mismatch\'):\n            c=next(c for c in d.build_cases(cfg) if c[\'scenario\']==scenario and c[\'regime\']==\'full\')\n            row=d.run_batch([c],\'random_safe\',cfg)[0][0]\n            self.assertEqual(row[\'guarantee_applicable\'],scenario==\'closed\')\n            if scenario in (\'off_grid\',\'missing_family\'): self.assertFalse(row[\'true_mean_in_catalog\'])\n\n    def test_error_coverage_smoke_under_adaptive_sampling(self):\n        # Regression gate, not a proof and not a publication-level calibration study.\n        cfg=d.Config(preset=\'smoke\',budget=24,eig_samples=4,game_iterations=8,device=\'cpu\',trace_per_group=0)\n        template=next(c for c in d.build_cases(cfg) if c[\'domain\']==\'kinetics\' and c[\'regime\']==\'full\')\n        cases=[]\n        for rep in range(200):\n            c=dict(template,rep=rep); c[\'id\']=d.digest([template[\'id\'],\'coverage-test\',rep])[:24]; cases.append(c)\n        rows,_=d.run_batch(cases,\'deficit_safe\',cfg)\n        errors=sum(r[\'false_declaration\'] for r in rows)\n        self.assertLessEqual(errors,20) # conservative regression alarm, alpha=.05\n\nif __name__==\'__main__\': unittest.main(verbosity=2)\n', 'PROTOCOL.md': '# Protocolo experimental — identificação ativa sob restrições\n\nVersão 1.0, 01/10/2026. Este é um piloto de pesquisa executável. Novidade e competitividade editorial ainda dependem de resultados e comparação externa.\n\n## Pergunta e contribuição candidata\n\nQuanto custa controlar falsas declarações de descoberta quando o agente seleciona as medições, os experimentos têm custos diferentes e algumas intervenções não estão disponíveis?\n\nO estudo separa três fatores: aquisição de dados, regra de declaração e capacidade do catálogo de representar a verdade. A proposta exploratória de aquisição dá prioridade às alternativas que ainda precisam de evidência para serem eliminadas. Sua eficiência será comparada com métodos simples e desenho por informação mútua. Não se pressupõe que ela vencerá.\n\nA unidade de inferência é a **família de equações em um catálogo finito**, com parâmetros discretizados conhecidos como possibilidades. Não há geração livre de novas equações no núcleo desta versão. Isso torna verificáveis as afirmações estatísticas e as equivalências. O painel LLM escolhe experimentos; ele não gera a verdade nem julga o resultado.\n\nEste recorte é mais estreito que descoberta científica geral. Resultados positivos aqui justificam a etapa seguinte, não uma alegação de resolver descoberta irrestrita.\n\n## O que mudou após a pesquisa adicional\n\n- [Universal inference, 2020](https://pmc.ncbi.nlm.nih.gov/articles/PMC7382245/) já fornece a base para testes e conjuntos de confiança sequenciais baseados em razões de verossimilhança. A validade estatística empregada aqui é uma aplicação desse princípio, não um teorema novo reivindicado.\n- [Deep Adaptive Design, ICML 2021](https://proceedings.mlr.press/v139/foster21a.html) já estuda políticas de desenho amortizadas. Treinar uma rede para escolher experimentos também não bastaria como novidade.\n- [AutoSciLab](https://arxiv.org/abs/2605.24043) e [MDA](https://arxiv.org/abs/2608.09696) são concorrentes externos necessários em uma versão ampliada. As políticas implementadas aqui não são reproduções desses sistemas.\n- [Chernoff Sampling, AISTATS 2022](https://proceedings.mlr.press/v151/mukherjee22a.html) é antecedente do desenho por discriminação. Nossa implementação é uma aproximação simples de alocação, identificada como tal.\n\n## Hipóteses a testar\n\nH1: o critério sequencial mantém o risco de declaração falsa no nível especificado quando verdade, catálogo e ruído satisfazem as condições.\n\nH2: aquisição sensível à evidência que falta pode reduzir custo de resolução correta em relação a EIG e alocação discriminativa convencional. H2 pode ser refutada pelos dados.\n\nH3: restrições nas intervenções tornam necessária uma resposta de equivalência em parte das instâncias; forçar uma única família pode produzir excesso de conclusões.\n\nH4: retirar a verdade do catálogo ou especificar o ruído incorretamente rompe a aplicabilidade da garantia. O experimento mede a gravidade dessa falha; não promete corrigi-la nesta versão.\n\n## Dados e ambientes abertos\n\nTodos os dados são gerados pelo código distribuído, sem arquivos privados, avaliação humana, serviço de API pago ou coleta externa. As leis são exemplos científicos simplificados, não dados experimentais reais. O catálogo completo é público para todos os métodos; o índice da lei verdadeira e as respostas futuras ficam no simulador/avaliador.\n\n**Mecânica simplificada:** seis formas com termos lineares, aditivos, cúbicos, de interação e quadráticos; 48 modelos parametrizados. Valores dos parâmetros: 1/2, 1 e 2. Menu completo com 36 intervenções racionais. Menu restrito: z=x, seis intervenções.\n\n**Cinética simplificada:** Michaelis–Menten, inibição competitiva, não competitiva, incompetitiva, inibição por substrato e Hill com expoente 2; 54 modelos parametrizados. Menu completo com 25 intervenções; restrito a inibidor z=0, cinco intervenções. As formas são controladas e não pretendem cobrir toda a bioquímica.\n\nMedições no eixo de restrição custam 1; intervenções fora dele custam 3. São custos de benchmark declarados, não preços de laboratório. A escala de medição é pública e fixa por domínio. O ruído padrão é gaussiano independente, com desvio conhecido na unidade normalizada.\n\nAs equivalências são calculadas com aritmética racional exata em **todo o menu finito de ações permitido**. Nenhuma tolerância numérica define as classes. A conclusão vale para esse menu; não constitui prova de equivalência em todo um domínio contínuo.\n\n## Regra sequencial e seu limite\n\nSeja H o catálogo finito e p_h(y|a) a densidade gaussiana da hipótese h na intervenção a. O agente escolhe a_t usando apenas informações anteriores. O prior público pi tem massa positiva em todas as hipóteses.\n\nDefinimos L_t(h)=produto_s p_h(y_s|a_s), Q_t=soma_h pi(h)L_t(h) e E_t(h)=Q_t/L_t(h). A computação usa logaritmos e float64. Constantes gaussianas comuns cancelam exatamente na expressão matemática.\n\nSob uma hipótese verdadeira h*, Q_t é a densidade sequencial da mistura: a seleção adaptativa de ações é previsível e usa a mesma política sob as hipóteses. Portanto E_t(h*) é um martingale não negativo iniciado em 1. Pela desigualdade de Ville, P(sup_t E_t(h*) >= 1/alpha) <= alpha.\n\nO conjunto C_t mantém hipóteses cujo E_s nunca cruzou 1/alpha até t. O sistema declara:\n\n1. **family:** todas as hipóteses sobreviventes têm a mesma família;\n2. **equivalence:** há mais de uma família sobrevivente, mas todas pertencem à mesma classe de respostas no menu;\n3. **unresolved:** o orçamento terminou sem as condições anteriores;\n4. **empty_catalog:** nenhuma hipótese sobreviveu. Isso sinaliza incompatibilidade com o procedimento, não prova um mecanismo externo específico.\n\nSe h* sobrevive, uma declaração family ou equivalence não pode excluir a verdade no sentido definido acima. Logo a probabilidade de uma declaração errada durante a execução é limitada por alpha nas condições especificadas. Não é necessário dividir alpha pelo tamanho do catálogo para a cobertura da única hipótese verdadeira; isso não deve ser confundido com controle de outras famílias de descobertas simultâneas.\n\nA garantia é por execução, não a probabilidade de zero erros em milhares de execuções. Também não implica que a fração de erros condicionada às declarações seja <= alpha. Ambas as métricas devem ser distinguidas.\n\n**Condições:** verdade exatamente representada, ruído corretamente especificado, ausência de acesso ao futuro, catálogo/prior fixados e cálculo fiel. A discretização não aproxima uma garantia contínua automaticamente. Hiperparâmetros escolhidos após olhar o teste também não devem ser vendidos como protocolo confirmatório independente.\n\n## Políticas e controles\n\n| Método | Aquisição | Declaração |\n|---|---|---|\n| random_safe | Ação acessível aleatória | Critério sequencial |\n| variance_safe | Variância posterior por custo | Critério sequencial |\n| eig_safe | Informação mútua Monte Carlo por custo, sobre o átomo do catálogo | Critério sequencial |\n| chernoff_safe | Jogo aproximado entre alocação e rival mais difícil; divergência gaussiana por custo | Critério sequencial |\n| deficit_safe | Mesmo jogo, ponderando pela evidência que falta para eliminar rivais sobreviventes | Critério sequencial |\n| eig_posterior | EIG | Probabilidade posterior de família/classe >= 1-alpha |\n| deficit_posterior | Aquisição deficit | Mesmo limiar posterior |\n| eig_fixed | EIG até o fim | Família MAP forçada no orçamento final |\n\nO controle eig_fixed é deliberadamente uma regra forçada, útil para diagnóstico; não deve representar todos os métodos científicos existentes. O limiar posterior não possui automaticamente a garantia frequentista uniforme acima.\n\nDeficit divide a separação esperada gaussiana para cada rival pelo déficit atual de log-evidência. A otimização usa atualizações multiplicativas aproximadas e exploração uniforme de 2%, também presente no controle chernoff. O código não reivindica solução ótima do jogo. A ablação crucial é deficit_safe versus chernoff_safe, além de eig_safe.\n\nO painel opcional usa [Qwen2.5-Coder-7B-Instruct](https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct) em inferência. Ele recebe histórico, ações/custos, desvio do ruído e oito candidatos com maior pontuação. Responde com um índice de ação. Respostas inválidas usam EIG, com registro explícito da ocorrência. Os mesmos episódios têm controles EIG e deficit. A amostra é pequena e exploratória.\n\n## Cenários e amostragem\n\n- closed: verdade no catálogo, ruído correto, intervenções completas ou restritas.\n- off_grid: coeficientes verdadeiros multiplicados por fatores racionais fora da grade, apenas menu completo.\n- missing_family: família verdadeira retirada do catálogo, apenas menu completo.\n- noise_mismatch: ruído real duas vezes maior que o informado, apenas menu completo.\n\nOs cenários de estresse são reportados separadamente; não há promessa de controle de erro neles. Para off_grid, correct significa recuperação da família estrutural, não dos coeficientes verdadeiros.\n\nPresets:\n\n- smoke: um parâmetro selecionado por família, uma repetição, um ruído, dois domínios e dois menus; 24 episódios por método.\n- pilot: dois parâmetros por família, oito repetições, dois ruídos, dois menus closed e três estresses; 1.920 episódios por método, 15.360 execuções no total.\n- confirmatory: mesma estrutura com 100 repetições; 24.000 episódios por método, 192.000 execuções. Requer múltiplas sessões conforme tempo medido; não há promessa de completar numa sessão gratuita.\n\nAs seleções de coeficientes, ruído e aleatoriedade das políticas têm sementes rastreáveis. Regras de declaração com a mesma política compartilham a fita aleatória, permitindo comparações pareadas. Alterar o batch não altera os episódios. Não treinamos políticas no conjunto de teste.\n\nO preset confirmatory é uma configuração de amostra maior. Antes de usá-lo como estudo confirmatório real, congelar código, decisões e hipóteses após o piloto, escolher uma nova semente e registrar o plano. O nome do preset não realiza pré-registro.\n\n## Análises e leitura dos resultados\n\nMedidas: declaração falsa por execução, resolução correta, tipo de resposta, custo, número de consultas, eliminação da verdade e tempo amortizado por batch. O tempo inclui aquisição e o bookkeeping do batch; não é latência isolada de uma chamada em GPU.\n\nCada contexto e lei recebe intervalo binomial exato para erros entre repetições independentes. Comparações entre métodos usam episódios pareados e bootstrap por lei. O agregado por lei dá peso igual às leis disponíveis; não é exatamente a mesma estimativa que agrupar todos os episódios indiscriminadamente. O gráfico agregado é descritivo.\n\nOs controles têm os mesmos custos de medição e número máximo de unidades. Custos de computação podem diferir; reportar tempos e configurações. Comparar custo junto com resolução correta: recusar tudo custa pouco.\n\nNenhum texto de conclusão científica é gerado automaticamente. Não selecionar somente as figuras favoráveis. O painel LLM e os testes de software devem permanecer separados do estudo principal.\n\n## Reprodução e limitações de Colab\n\nO núcleo usa tensores float64 e simulações em batch. Não requer treinamento de grande modelo. A GPU acelera os cálculos de informação e os jogos de alocação. O código detecta CUDA; em CPU funciona com o preset smoke. A memória da GPU depende principalmente de batch × amostras EIG × ações × hipóteses. Batch 32 é conservador para o catálogo atual.\n\nO modelo de 7B usa BF16/FP16 quando há pelo menos 23 GiB livres; caso contrário, tenta NF4 de quatro bits. A instalação de dependências não substitui o PyTorch do Colab. A revisão do modelo, hardware e versões são registradas.\n\nO [Colab não garante GPUs específicas no plano gratuito](https://research.google.com/colaboratory/faq.html). Uma A100 disponível é suficiente como alvo de engenharia para esse painel, mas a execução em A100 ainda precisa ser medida. Há checkpoints atômicos, retomada por episódio e backup periódico opcional no Drive. Nenhuma chave de API é necessária.\n\n## O que falta para uma submissão forte\n\n1. Executar o piloto e testar se H2 sobrevive às comparações; abandonar a política se não agregar.\n2. Ampliar famílias, custos e parâmetros, incluindo sensibilidade à discretização.\n3. Integrar pelo menos um simulador externo e reproduzir concorrentes recentes com protocolo compatível.\n4. Estudar hipóteses compostas com parâmetros contínuos; o teorema discreto não se estende por conveniência.\n5. Definir uma contribuição científica original verificável além da aplicação de inferência sequencial já conhecida.\n\nO pacote entrega a infraestrutura, um desenho falsificável e verificações do núcleo. Não entrega evidência de superioridade nem garante aceitação top tier.\n', 'README.md': '# Experimento de descoberta ativa — Colab\n\n[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JPSchettino/model_analisys-/blob/active-mechanism-discovery/research/active_mechanism_discovery/Discovery_Colab.ipynb)\n\nO notebook principal busca os arquivos de um commit fixo deste repositório e verifica SHA-256 antes de executá-los. Abra pelo botão acima. A alternativa `Discovery_Colab_standalone.ipynb` contém as fontes e pode ser enviada pelo menu Arquivo → Fazer upload de notebook.\n\n1. Selecione um runtime com GPU. Se houver A100, use-a; ela não é garantida no plano gratuito.\n2. Execute as células em ordem. O padrão `pilot` faz o experimento principal. `smoke` é uma verificação rápida.\n3. Autorize a montagem do seu Drive se quiser backup entre sessões. Sem isso, os resultados ficam no runtime até o download.\n4. Ao final, o notebook gera gráficos, tabelas e um ZIP. Envie esse ZIP para analisarmos o resultado.\n5. Se a execução parar pelo limite configurado ou desconectar, abra o mesmo notebook, mantenha a configuração e execute novamente. O backup do Drive restaura o último snapshot.\n\nO padrão contém oito métodos, dois domínios, ruído e intervenções restritas, além de estresses em que a garantia não se aplica. O painel opcional de 7B usa pesos abertos e nenhuma API paga. Ele pode ser desligado com `RUN_LLM_PANEL=False`.\n\n**Esta versão estuda identificação de mecanismos em um catálogo finito.** As equivalências são exatas dentro do menu de intervenções definido. O LLM não avalia resultados. Consulte PROTOCOL.md para hipóteses, regra estatística, comparações e limites.\n\nArquivos gerados:\n\n- `summary.csv`: resultados por domínio, contexto, ruído e método\n- `risk_by_instance.csv`: intervalos para taxas de erro por lei\n- `paired_comparisons.csv`: comparações pareadas com EIG, bootstrap por lei\n- `main_results.png` e `.pdf`: gráficos descritivos\n- `episodes.csv`: dados completos para reanálise\n- `manifest.json`, `status.json`, `runtime_packages.txt`: configuração, progresso e ambiente\n- `llm_panel/`: painel separado, prompts, respostas, revisão dos pesos e ocorrências de fallback\n- `PROTOCOL.md` e fontes do código: cópia junto com os resultados\n\nSem Colab, com dependências instaladas: `python discovery.py --preset smoke --out results --device cpu`.\n\nOs resultados locais de `validation_cpu` validam o funcionamento do software. Não constituem a amostra principal do artigo. A execução CUDA e o carregamento real dos pesos precisam ser confirmados no Colab.\n'}
EXPECTED_SHA256 = {'discovery.py': 'e15b083445e970b7dccdc3081efe9cbcd9987e24434dc6d6f56468e2976f22d5', 'llm_panel.py': 'b7478bca3e0f4448d24da871bf39dbd73ffcabf85c9ee644648434c4b188b4b1', 'test_discovery.py': '7f5cc6e53d99cb2e18242e49f2bd066698c4d17b4699d69454a068d02a2465f1', 'PROTOCOL.md': '80aa87461ae8bbd0f0ec5fb21b70d756dd83e92a474dbd5576cd33d3b646be34', 'README.md': '13b48b970d9e34ed1251500134033456c2a2f8b535887e300b293de97001f4f1'}

import hashlib
ROOT=Path(os.environ.get('DISCOVERY_ROOT','/content/mechanism_discovery'))
ROOT.mkdir(parents=True,exist_ok=True)
for name,content in EMBEDDED_FILES.items():
    assert hashlib.sha256(content.encode()).hexdigest()==EXPECTED_SHA256[name]
    (ROOT/name).write_text(content)
sys.path.insert(0,str(ROOT))
import discovery
print('Código materializado e verificado em',ROOT)


## Configuração

`smoke`: 192 execuções pequenas. `pilot`: 15.360 execuções, em blocos com retomada. `confirmatory`: 192.000 execuções; requer protocolo congelado, nova semente e possivelmente várias sessões.

`MAX_MINUTES` limita cada chamada do núcleo. Execute novamente a célula principal para continuar. O núcleo usa float64 para a evidência. O painel opcional usa um modelo aberto de 7B, com precisão ajustada à memória livre.


In [ ]:
PRESET = 'pilot'  # 'smoke', 'pilot', 'confirmatory'
SEED = 20261001 if PRESET != 'confirmatory' else 20261107
BUDGET = 60
ALPHA = 0.05
BATCH = 32
MAX_MINUTES = 45
PERSIST_DRIVE = True
RUN_LLM_PANEL = True
LLM_CASES = 12
LLM_BUDGET = 24

from discovery import Config, build_cases, digest
cfg=Config(preset=PRESET,seed=SEED,budget=BUDGET,alpha=ALPHA,batch=BATCH,max_minutes=MAX_MINUTES)
import dataclasses
scientific_config=dataclasses.asdict(cfg)
for key in ('batch','device','max_minutes','trace_per_group'): scientific_config.pop(key)
run_id=PRESET+'_'+digest({'config':scientific_config,'source':EXPECTED_SHA256['discovery.py']})[:12]
RUN_DIR=ROOT/'runs'/run_id
RUN_DIR.mkdir(parents=True,exist_ok=True)
print('Execução:',run_id,'| episódios × métodos:',len(build_cases(cfg))*len(cfg.methods))


In [ ]:
import json, time, zipfile, shutil
DRIVE_BACKUP = None
if PERSIST_DRIVE:
    try:
        from google.colab import drive
    except ImportError:
        print('Drive indisponível fora do Colab; usando armazenamento local.')
    else:
        drive.mount('/content/drive')
        backup_root=Path('/content/drive/MyDrive/Mechanism_Discovery_runs')
        backup_root.mkdir(parents=True,exist_ok=True)
        DRIVE_BACKUP=backup_root/(run_id+'.zip')
        if DRIVE_BACKUP.exists() and not (RUN_DIR/'manifest.json').exists():
            with zipfile.ZipFile(DRIVE_BACKUP) as z:
                for member in z.infolist():
                    if not (RUN_DIR/member.filename).resolve().is_relative_to(RUN_DIR.resolve()):
                        raise ValueError('Caminho inválido no backup.')
                z.extractall(RUN_DIR)
            print('Backup restaurado:',DRIVE_BACKUP.name)

last_backup=[0.0]
def checkpoint(folder,force=False):
    if DRIVE_BACKUP is None: return
    if not force and time.monotonic()-last_backup[0]<120: return
    local_zip=ROOT/'checkpoint_staging.zip'
    with zipfile.ZipFile(local_zip,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
        for p in Path(folder).rglob('*'):
            if p.is_file() and not p.name.endswith('.tmp'):
                z.write(p,p.relative_to(folder))
    pending=DRIVE_BACKUP.with_suffix('.pending')
    shutil.copyfile(local_zip,pending)
    os.replace(pending,DRIVE_BACKUP)
    last_backup[0]=time.monotonic()

for name in EMBEDDED_FILES:
    (RUN_DIR/name).write_text(EMBEDDED_FILES[name])
print('Backup periódico:',str(DRIVE_BACKUP) if DRIVE_BACKUP else 'desligado — baixe os resultados antes de sair')


## Verificação antes de gastar a sessão

Testes do cálculo da evidência, equivalência exata, respeito ao orçamento, retomada, independência do tamanho do batch e um teste estatístico de regressão. Estes testes verificam o software; não substituem a amostra científica.


In [ ]:
import unittest, test_discovery
suite=unittest.defaultTestLoader.loadTestsFromModule(test_discovery)
result=unittest.TextTestRunner(verbosity=1).run(suite)
assert result.wasSuccessful(), 'Corrija o erro antes do estudo principal.'


## Experimento principal

Pode ser reexecutado com a mesma configuração. Episódios concluídos são reutilizados. Alterações científicas ou de código criam outra pasta. Um limite de tempo salva o progresso e permite continuar depois.


In [ ]:
from discovery import run_suite, summarize
started=time.monotonic()
run_suite(cfg,RUN_DIR,checkpoint=checkpoint)
print('Minutos nesta chamada:',round((time.monotonic()-started)/60,2))


In [ ]:
summary=summarize(RUN_DIR)
display(summary)
from IPython.display import Image, display
display(Image(filename=str(RUN_DIR/'main_results.png')))
status=json.loads((RUN_DIR/'status.json').read_text())
print('Núcleo completo:',status['completed'])
if not status['completed']:
    print('Resultados parciais. Reexecute a célula principal para continuar.')
checkpoint(RUN_DIR,force=True)


## Painel exploratório com LLM aberto

O modelo sugere a próxima ação, recebe apenas informações disponíveis ao agente e não atribui notas. Os resultados ficam separados. Baixa cerca de 15 GB de pesos no modo sem quantização; tempo e memória dependem do runtime. Respostas inválidas usam uma política de fallback explicitamente registrada.

O painel requer o núcleo completo e GPU. A revisão dos pesos é fixada no manifesto. Se esta parte falhar por download ou memória, os resultados do núcleo continuam salvos; o erro é registrado e os arquivos podem ser exportados na célula seguinte.


In [ ]:
if RUN_LLM_PANEL and status['completed'] and torch.cuda.is_available():
    try:
        subprocess.run([sys.executable,'-m','pip','install',
            'transformers>=4.57,<6','accelerate>=1,<2','bitsandbytes>=0.48,<1',
            'huggingface_hub>=0.34'],check=True)
        from llm_panel import run_panel
        display(run_panel(RUN_DIR/'llm_panel',max_cases=LLM_CASES,budget=LLM_BUDGET,seed=SEED,checkpoint=checkpoint))
    except Exception as exc:
        discovery.atomic_json(RUN_DIR/'llm_panel_error.json',
            {'type':type(exc).__name__,'message':str(exc),'completed':False})
        print('Painel LLM incompleto:',type(exc).__name__,str(exc))
        print('O núcleo está salvo. Você pode exportar os arquivos e depois retomar o painel.')
    finally:
        checkpoint(RUN_DIR,force=True)
else:
    print('Painel não executado: confira RUN_LLM_PANEL, conclusão do núcleo e GPU.')


## Exportar tudo

O ZIP contém tabelas, gráficos, episódios, fontes, protocolo, ambiente e painel LLM quando executado. Não inclui pesos de modelo. Envie esse ZIP para análise; gráficos favoráveis isolados não bastam para concluir superioridade.


In [ ]:
freeze=subprocess.run([sys.executable,'-m','pip','freeze'],capture_output=True,text=True,check=True).stdout
(RUN_DIR/'runtime_packages.txt').write_text(freeze)
discovery.atomic_json(RUN_DIR/'final_environment.json',discovery.environment())
checkpoint(RUN_DIR,force=True)
export_path=ROOT/(run_id+'_resultados.zip')
with zipfile.ZipFile(export_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in RUN_DIR.rglob('*'):
        if p.is_file() and not p.name.endswith('.tmp'):
            z.write(p,p.relative_to(RUN_DIR))
print('Pronto:',export_path)
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(str(export_path))


## Antes de interpretar

- `closed` testa o procedimento com verdade no catálogo e ruído correto. Os outros cenários não recebem a mesma garantia.
- `family` identifica uma forma, sem certificar seus coeficientes contínuos. `equivalence` vale para o menu finito de intervenções.
- Intervalos de erro estão em `risk_by_instance.csv`. O gráfico agregado é descritivo.
- Uma taxa baixa de erro com pouca resolução não estabelece utilidade. Compare as duas e o custo.
- A validade estatística usa resultados conhecidos de inferência sequencial. A novidade candidata está no problema experimental e na aquisição orientada à evidência; precisa ser demonstrada.
- As variantes simples deste notebook não substituem a comparação externa com AutoSciLab/MDA. Consulte `PROTOCOL.md` para o plano de ampliação.
